In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import BorderlineSMOTE
from imblearn.combine import SMOTETomek
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

In [2]:
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv(
    "/content/drive/MyDrive/ciciot2023_consolidado.csv",
    low_memory=False,
    nrows=2000000
)
print(f"Shape: {df.shape}")
print(f"Clases unicas: {df['label'].nunique()}")
print(df['label'].value_counts())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Shape: (2000000, 47)
Clases unicas: 34
label
DDoS-ICMP_Flood            309059
DDoS-UDP_Flood             233026
DDoS-TCP_Flood             192340
DDoS-PSHACK_Flood          175321
DDoS-SYN_Flood             173111
DDoS-RSTFINFlood           172808
DDoS-SynonymousIP_Flood    154546
DoS-UDP_Flood              141532
DoS-TCP_Flood              114097
DoS-SYN_Flood               86695
BenignTraffic               46939
Mirai-greeth_flood          42456
Mirai-udpplain              38400
Mirai-greip_flood           32533
DDoS-ICMP_Fragmentation     19159
MITM-ArpSpoofing            13147
DDoS-UDP_Fragmentation      12583
DDoS-ACK_Fragmentation      12388
DNS_Spoofing                 7766
Recon-HostDiscovery          5704
Recon-OSScan                 4215
Recon-PortScan               3421
DoS-HTTP_Flood               3148
VulnerabilityScan            1574
DDoS-HTTP_

## Eliminar duplicados y separar features/label

In [3]:
df = df.drop_duplicates()
print(f"Shape despues de eliminar duplicados: {df.shape}")

X = df.drop(columns=['label'])
y = df['label']

print(f"Features: {X.shape[1]}")
print(f"Clases: {y.nunique()}")

Shape despues de eliminar duplicados: (2000000, 47)
Features: 46
Clases: 34


## Codificar etiquetas

In [4]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print("Mapeo de clases:")
for i, clase in enumerate(le.classes_):
    print(f"  {i:2d} -> {clase}")

Mapeo de clases:
   0 -> Backdoor_Malware
   1 -> BenignTraffic
   2 -> BrowserHijacking
   3 -> CommandInjection
   4 -> DDoS-ACK_Fragmentation
   5 -> DDoS-HTTP_Flood
   6 -> DDoS-ICMP_Flood
   7 -> DDoS-ICMP_Fragmentation
   8 -> DDoS-PSHACK_Flood
   9 -> DDoS-RSTFINFlood
  10 -> DDoS-SYN_Flood
  11 -> DDoS-SlowLoris
  12 -> DDoS-SynonymousIP_Flood
  13 -> DDoS-TCP_Flood
  14 -> DDoS-UDP_Flood
  15 -> DDoS-UDP_Fragmentation
  16 -> DNS_Spoofing
  17 -> DictionaryBruteForce
  18 -> DoS-HTTP_Flood
  19 -> DoS-SYN_Flood
  20 -> DoS-TCP_Flood
  21 -> DoS-UDP_Flood
  22 -> MITM-ArpSpoofing
  23 -> Mirai-greeth_flood
  24 -> Mirai-greip_flood
  25 -> Mirai-udpplain
  26 -> Recon-HostDiscovery
  27 -> Recon-OSScan
  28 -> Recon-PingSweep
  29 -> Recon-PortScan
  30 -> SqlInjection
  31 -> Uploading_Attack
  32 -> VulnerabilityScan
  33 -> XSS


## Aplicar RobustScaler

In [5]:
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print("RobustScaler aplicado correctamente")
print(f"Shape: {X_scaled.shape}")
print(f"\nEstadisticas post-escalado (primeras 5 features):")
print(X_scaled.iloc[:, :5].describe().round(4))

RobustScaler aplicado correctamente
Shape: (2000000, 46)

Estadisticas post-escalado (primeras 5 features):
       flow_duration  Header_Length  Protocol Type      Duration          Rate
count   2.000000e+06   2.000000e+06   2.000000e+06  2.000000e+06  2.000000e+06
mean    5.427180e+01   2.880904e+02   3.507000e-01  2.329800e+00  7.631810e+01
std     2.586024e+03   1.732263e+03   1.021100e+00  1.391710e+01  8.428735e+02
min     0.000000e+00  -2.028000e-01  -6.840000e-01 -6.400000e+01 -1.352000e-01
25%     0.000000e+00   0.000000e+00   0.000000e+00  0.000000e+00 -1.173000e-01
50%     0.000000e+00   0.000000e+00   0.000000e+00  0.000000e+00  0.000000e+00
75%     1.000000e+00   1.000000e+00   1.000000e+00  0.000000e+00  8.827000e-01
max     8.870202e+05   3.707644e+04   4.673700e+00  1.910000e+02  7.155816e+04


## Estrategia de manejo del desbalance


In [6]:
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler

conteo_original = Counter(y_encoded)
print("Distribucion original (top 10):")
for clase, count in sorted(conteo_original.items(), key=lambda x: -x[1])[:10]:
    print(f"  {le.classes_[clase]}: {count:,}")

LIMITE_MAXIMO = 100000
LIMITE_MINIMO = 1000

under_strategy = {k: min(v, LIMITE_MAXIMO) for k, v in conteo_original.items()}
rus = RandomUnderSampler(sampling_strategy=under_strategy, random_state=42)
X_under, y_under = rus.fit_resample(X_scaled, y_encoded)

conteo_under = Counter(y_under)
print(f"\nShape tras undersampling: {X_under.shape}")

over_strategy = {k: max(v, LIMITE_MINIMO) for k, v in conteo_under.items()}
ros = RandomOverSampler(sampling_strategy=over_strategy, random_state=42)
X_bal, y_bal = ros.fit_resample(X_under, y_under)

print(f"Shape tras oversampling: {X_bal.shape}")
print(f"\nDistribucion final:")
conteo_final = Counter(y_bal)
for clase, count in sorted(conteo_final.items(), key=lambda x: -x[1]):
    print(f"  {le.classes_[clase]}: {count:,}")

Distribucion original (top 10):
  DDoS-ICMP_Flood: 309,059
  DDoS-UDP_Flood: 233,026
  DDoS-TCP_Flood: 192,340
  DDoS-PSHACK_Flood: 175,321
  DDoS-SYN_Flood: 173,111
  DDoS-RSTFINFlood: 172,808
  DDoS-SynonymousIP_Flood: 154,546
  DoS-UDP_Flood: 141,532
  DoS-TCP_Flood: 114,097
  DoS-SYN_Flood: 86,695

Shape tras undersampling: (1234160, 46)
Shape tras oversampling: (1240420, 46)

Distribucion final:
  DDoS-ICMP_Flood: 100,000
  DDoS-PSHACK_Flood: 100,000
  DDoS-RSTFINFlood: 100,000
  DDoS-SYN_Flood: 100,000
  DDoS-SynonymousIP_Flood: 100,000
  DDoS-TCP_Flood: 100,000
  DDoS-UDP_Flood: 100,000
  DoS-TCP_Flood: 100,000
  DoS-UDP_Flood: 100,000
  DoS-SYN_Flood: 86,695
  BenignTraffic: 46,939
  Mirai-greeth_flood: 42,456
  Mirai-udpplain: 38,400
  Mirai-greip_flood: 32,533
  DDoS-ICMP_Fragmentation: 19,159
  MITM-ArpSpoofing: 13,147
  DDoS-UDP_Fragmentation: 12,583
  DDoS-ACK_Fragmentation: 12,388
  DNS_Spoofing: 7,766
  Recon-HostDiscovery: 5,704
  Recon-OSScan: 4,215
  Recon-PortScan: 3

## Train/test split

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_bal, y_bal,
    test_size=0.2,
    random_state=42,
    stratify=y_bal
)

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train: {Counter(y_train)}")

X_train: (992336, 46)
X_test:  (248084, 46)
y_train: Counter({np.int64(13): 80000, np.int64(21): 80000, np.int64(12): 80000, np.int64(10): 80000, np.int64(8): 80000, np.int64(6): 80000, np.int64(9): 80000, np.int64(20): 80000, np.int64(14): 80000, np.int64(19): 69356, np.int64(1): 37551, np.int64(23): 33965, np.int64(25): 30720, np.int64(24): 26026, np.int64(7): 15327, np.int64(22): 10518, np.int64(15): 10066, np.int64(4): 9910, np.int64(16): 6213, np.int64(26): 4563, np.int64(27): 3372, np.int64(29): 2737, np.int64(18): 2519, np.int64(32): 1259, np.int64(5): 992, np.int64(11): 842, np.int64(3): 800, np.int64(28): 800, np.int64(0): 800, np.int64(33): 800, np.int64(17): 800, np.int64(30): 800, np.int64(2): 800, np.int64(31): 800})


## Guardar datasets procesados en Drive

In [8]:
import numpy as np

np.save("/content/drive/MyDrive/X_train.npy", X_train)
np.save("/content/drive/MyDrive/X_test.npy", X_test)
np.save("/content/drive/MyDrive/y_train.npy", y_train)
np.save("/content/drive/MyDrive/y_test.npy", y_test)
np.save("/content/drive/MyDrive/feature_names.npy", np.array(X.columns.tolist()))

import pickle
with open("/content/drive/MyDrive/label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)
with open("/content/drive/MyDrive/robust_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

print("Archivos guardados en Drive:")
print("  X_train.npy, X_test.npy")
print("  y_train.npy, y_test.npy")
print("  feature_names.npy")
print("  label_encoder.pkl")
print("  robust_scaler.pkl")

Archivos guardados en Drive:
  X_train.npy, X_test.npy
  y_train.npy, y_test.npy
  feature_names.npy
  label_encoder.pkl
  robust_scaler.pkl
